### ```USE Stratified Shuffle Split in Scikit-Learn```

In [1]:
import numpy as np
import pandas as pd

In [2]:
import matplotlib.pyplot as plt

In [3]:
df = pd.read_csv("05_housing.csv")

In [4]:
df['income_cat'] = pd.cut(df['median_income'], bins=[0, 1.5, 3.0, 4.5, 6.0, np.inf], labels=[1, 2, 3, 4, 5])

In [5]:
from sklearn.model_selection import StratifiedShuffleSplit
split = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
for train_index, test_index in split.split(df, df['income_cat']):
    strat_train_set = df.loc[train_index]
    strat_test_set = df.loc[test_index]
    

In [6]:
# lets remove the income_cat colum
for sett in (strat_train_set,strat_test_set):
    sett.drop("income_cat",axis = 1, inplace= True)
    

In [7]:
df = strat_train_set.copy()

In [8]:
# df

In [9]:
housing = strat_train_set.drop("median_house_value",axis = 1)
housing_labels = strat_train_set["median_house_value"].copy() 

In [10]:
# housing

## ```Handling Missing Data```
- use SimpleImputer functon to handle the missing data
- SimpleImputer is a sklearn library use for handling missing data

In [11]:
from sklearn.impute import SimpleImputer

In [12]:
imputer = SimpleImputer(strategy  = 'median')

In [13]:
housing_num = housing.select_dtypes(include=[np.number])  # only sleect the numerical values column

In [14]:
imputer.fit(housing_num)

,missing_values,nan
,strategy,'median'
,fill_value,None
,copy,True
,add_indicator,False
,keep_empty_features,False


In [15]:
X = imputer.transform(housing_num)

In [16]:
X

array([[-1.2146e+02,  3.8520e+01,  2.9000e+01, ...,  2.2370e+03,
         7.0600e+02,  2.1736e+00],
       [-1.1723e+02,  3.3090e+01,  7.0000e+00, ...,  2.0150e+03,
         7.6800e+02,  6.3373e+00],
       [-1.1904e+02,  3.5370e+01,  4.4000e+01, ...,  6.6700e+02,
         3.0000e+02,  2.8750e+00],
       ...,
       [-1.2272e+02,  3.8440e+01,  4.8000e+01, ...,  4.5800e+02,
         1.7200e+02,  3.1797e+00],
       [-1.2270e+02,  3.8310e+01,  1.4000e+01, ...,  1.2080e+03,
         5.0100e+02,  4.1964e+00],
       [-1.2214e+02,  3.9970e+01,  2.7000e+01, ...,  6.2500e+02,
         1.9700e+02,  3.1319e+00]], shape=(16512, 8))

In [17]:
# X - it givs numpy array , 

In [18]:
housing = pd.DataFrame(X,columns =housing_num.columns, index = housing_num.index)

In [19]:
housing

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income
12655,-121.46,38.52,29.0,3873.0,797.0,2237.0,706.0,2.1736
15502,-117.23,33.09,7.0,5320.0,855.0,2015.0,768.0,6.3373
2908,-119.04,35.37,44.0,1618.0,310.0,667.0,300.0,2.8750
14053,-117.13,32.75,24.0,1877.0,519.0,898.0,483.0,2.2264
20496,-118.70,34.28,27.0,3536.0,646.0,1837.0,580.0,4.4964
...,...,...,...,...,...,...,...,...
15174,-117.07,33.03,14.0,6665.0,1231.0,2026.0,1001.0,5.0900
12661,-121.42,38.51,15.0,7901.0,1422.0,4769.0,1418.0,2.8139
19263,-122.72,38.44,48.0,707.0,166.0,458.0,172.0,3.1797
19140,-122.70,38.31,14.0,3155.0,580.0,1208.0,501.0,4.1964


### ```add ocean_proximity in housing dataframe comlumn that we remove in previous```

In [20]:
housing['ocean_proximity'] = df['ocean_proximity']

In [21]:
housing = housing[['ocean_proximity']]

In [22]:
housing

,ocean_proximity
12655,INLAND
15502,NEAR OCEAN
2908,INLAND
14053,NEAR OCEAN
20496,<1H OCEAN
...,...
15174,<1H OCEAN
12661,INLAND
19263,<1H OCEAN
19140,<1H OCEAN


In [23]:
set( housing['ocean_proximity'])

{'<1H OCEAN', 'INLAND', 'ISLAND', 'NEAR BAY', 'NEAR OCEAN'}

## USE Ordinal Encoding 
- Scikit-Learn’s OrdinalEncoder can convert categories to numbers:

In [24]:
# from sklearn.preprocessing  import OrdinalEncoder
# ordinal_encoder = OrdinalEncoder()

In [25]:
# housing_cat = ordinal_encoder.fit_transform(housing)

In [26]:
# housing_cat

In [27]:
# housing_cat = pd.DataFrame(housing_cat,columns =housing.columns, index = housing.index)
# housing_cat

```⚠️ Caution: Ordinal encoding implies an order between categories, which may not ```
```be true here. For example, it treats INLAND (1) as closer to <1H OCEAN (0) than ```
```NEAR OCEAN (4) , which might not make sense```

## USE One-Hot Encoding
- ```For unordered categories, one-hot encoding is a better choice. It creates one binary column per category.```

In [28]:
from sklearn.preprocessing import OneHotEncoder

One_hot_encoder = OneHotEncoder()


In [29]:
housing_cat = One_hot_encoder.fit_transform(housing)

In [30]:
housing_cat.toarray()

array([[0., 1., 0., 0., 0.],
       [0., 0., 0., 0., 1.],
       [0., 1., 0., 0., 0.],
       ...,
       [1., 0., 0., 0., 0.],
       [1., 0., 0., 0., 0.],
       [0., 1., 0., 0., 0.]], shape=(16512, 5))

In [31]:
housing_cat = pd.DataFrame(housing_cat.toarray(),columns =['<1H OCEAN', 'INLAND', 'ISLAND', 'NEAR BAY', 'NEAR OCEAN'], index = housing.index)

In [32]:
df = pd.concat([df,housing_cat], axis = 1)
df

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity,<1H OCEAN,INLAND,ISLAND,NEAR BAY,NEAR OCEAN
12655,-121.46,38.52,29,3873,797.0,2237,706,2.1736,72100,INLAND,0.0,1.0,0.0,0.0,0.0
15502,-117.23,33.09,7,5320,855.0,2015,768,6.3373,279600,NEAR OCEAN,0.0,0.0,0.0,0.0,1.0
2908,-119.04,35.37,44,1618,310.0,667,300,2.8750,82700,INLAND,0.0,1.0,0.0,0.0,0.0
14053,-117.13,32.75,24,1877,519.0,898,483,2.2264,112500,NEAR OCEAN,0.0,0.0,0.0,0.0,1.0
20496,-118.70,34.28,27,3536,646.0,1837,580,4.4964,238300,<1H OCEAN,1.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15174,-117.07,33.03,14,6665,1231.0,2026,1001,5.0900,268500,<1H OCEAN,1.0,0.0,0.0,0.0,0.0
12661,-121.42,38.51,15,7901,1422.0,4769,1418,2.8139,90400,INLAND,0.0,1.0,0.0,0.0,0.0
19263,-122.72,38.44,48,707,166.0,458,172,3.1797,140400,<1H OCEAN,1.0,0.0,0.0,0.0,0.0
19140,-122.70,38.31,14,3155,580.0,1208,501,4.1964,258100,<1H OCEAN,1.0,0.0,0.0,0.0,0.0


In [33]:
df = df.drop("ocean_proximity", axis =1)
df

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,<1H OCEAN,INLAND,ISLAND,NEAR BAY,NEAR OCEAN
12655,-121.46,38.52,29,3873,797.0,2237,706,2.1736,72100,0.0,1.0,0.0,0.0,0.0
15502,-117.23,33.09,7,5320,855.0,2015,768,6.3373,279600,0.0,0.0,0.0,0.0,1.0
2908,-119.04,35.37,44,1618,310.0,667,300,2.8750,82700,0.0,1.0,0.0,0.0,0.0
14053,-117.13,32.75,24,1877,519.0,898,483,2.2264,112500,0.0,0.0,0.0,0.0,1.0
20496,-118.70,34.28,27,3536,646.0,1837,580,4.4964,238300,1.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15174,-117.07,33.03,14,6665,1231.0,2026,1001,5.0900,268500,1.0,0.0,0.0,0.0,0.0
12661,-121.42,38.51,15,7901,1422.0,4769,1418,2.8139,90400,0.0,1.0,0.0,0.0,0.0
19263,-122.72,38.44,48,707,166.0,458,172,3.1797,140400,1.0,0.0,0.0,0.0,0.0
19140,-122.70,38.31,14,3155,580.0,1208,501,4.1964,258100,1.0,0.0,0.0,0.0,0.0


## ```Feature Scaling and Transformation```          
- Feature scaling is a crucial preprocessing step. Most machine learning algorithms 
perform poorly when input features have vastly different scales.

#### Methods

### ```2.Min-Max Scaling (Normalization)```

In [34]:
# from sklearn.preprocessing import MinMaxScaler

In [35]:
# scaler = MinMaxScaler(feature_range=(-1,1))

In [36]:
# df_scaled = scaler.fit_transform(df)

In [37]:
# df_scaled = pd.DataFrame(df_scaled, columns = df.columns, index = df.index)
# df_scaled

- Use feature_range=(-1, 1) for models like neural networks. 
- Sensitive to outliers — extreme values can distort the scale.

### Inseted of ```Min-Max Scaling``` we use   ```Standardization (Z-score Scaling)```
- This method centers the data around 0 and scales it based on standard deviation. 


### FORMULA:```standardized_value = (x - mean) / std```

```Use Scikit-Learn’s StandardScaler :```

In [38]:
from sklearn.preprocessing import StandardScaler

In [39]:
scaler = StandardScaler()

In [43]:
df_scaled = scaler.fit_transform(df)

In [44]:
df_scaled = pd.DataFrame(df_scaled, columns = df.columns, index = df.index)
df_scaled

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,<1H OCEAN,INLAND,ISLAND,NEAR BAY,NEAR OCEAN
12655,-0.941350,1.347438,0.027564,0.584777,0.635123,0.732602,0.556286,-0.893647,-1.166015,-0.887683,1.46218,-0.011006,-0.354889,-0.384217
15502,1.171782,-1.192440,-1.722018,1.261467,0.775677,0.533612,0.721318,1.292168,0.627451,-0.887683,-0.68391,-0.011006,-0.354889,2.602693
2908,0.267581,-0.125972,1.220460,-0.469773,-0.545045,-0.674675,-0.524407,-0.525434,-1.074397,-0.887683,1.46218,-0.011006,-0.354889,-0.384217
14053,1.221738,-1.351474,-0.370069,-0.348652,-0.038567,-0.467617,-0.037297,-0.865929,-0.816829,-0.887683,-0.68391,-0.011006,-0.354889,2.602693
20496,0.437431,-0.635818,-0.131489,0.427179,0.269198,0.374060,0.220898,0.325752,0.270486,1.126529,-0.68391,-0.011006,-0.354889,-0.384217
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15174,1.251711,-1.220505,-1.165333,1.890456,1.686854,0.543471,1.341519,0.637374,0.531511,1.126529,-0.68391,-0.011006,-0.354889,-0.384217
12661,-0.921368,1.342761,-1.085806,2.468471,2.149712,3.002174,2.451492,-0.557509,-1.007844,-0.887683,1.46218,-0.011006,-0.354889,-0.384217
19263,-1.570794,1.310018,1.538566,-0.895802,-0.894007,-0.862013,-0.865118,-0.365475,-0.575684,1.126529,-0.68391,-0.011006,-0.354889,-0.384217
19140,-1.560803,1.249211,-1.165333,0.249005,0.109257,-0.189747,0.010616,0.168261,0.441622,1.126529,-0.68391,-0.011006,-0.354889,-0.384217
